# Trajectories

How you get from one pose to the next matters as much as the pose. This
module sends the same move three ways: as a **step** (the new pose at once),
as a **minimum-jerk** trajectory computed and streamed by your program, and
faster and faster until the robot's **rate limits** take over. It measures
how well the arm follows each one.

The arm's lag is measured in [Joints and frames](joints_and_frames.ipynb).
Run on the mock first (`HOST = "mock"`).

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME) if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

In [ ]:
class JointLog:
    """Records every joint reading (50 a second) while it is running:
    t (this computer's clock, when the reading arrived), measured q (29)
    and target q (29). Given c (your r.control()), target is what your
    program was sending for the arms at that moment; without it, what the
    robot reports it was asked for (the simulator reports nothing: NaN)."""
    def __init__(self, robot, c=None):
        self.rows, self._c = [], c
        self._stop = robot.on("joints", self._add)
    def _add(self, js):
        tgt = js.target if js.target is not None else np.full(len(js.q), np.nan)
        if self._c is not None:
            tgt = np.full(len(js.q), np.nan)
            for side, first in (("left", "L_sh_pitch"), ("right", "R_sh_pitch")):
                i = g1lab.BODY.index(first)
                tgt[i:i + 7] = self._c.arms.target(side)
        self.rows.append((time.time(), js.q.copy(), np.asarray(tgt, float).copy()))
    def stop(self, t0=None):
        """Stop recording: (t, q, target), t in s from t0 (default: the first reading)."""
        self._stop()
        t = np.array([row[0] for row in self.rows])
        return t - (t[0] if t0 is None else t0), np.array([row[1] for row in self.rows]), np.array([row[2] for row in self.rows])

In [ ]:
PITCH = g1lab.BODY.index("R_sh_pitch")       # the joint we move: right shoulder pitch
home = info.home["right_arm"].copy()
forward = home.copy()
forward[0] -= 0.8                            # 0.8 rad forward
RATE = 50                                    # how often we send, per second

def velocity(t, x):
    """Speed from samples (rad/s), smoothed over 3 readings."""
    v = np.gradient(x, t)
    return np.convolve(v, np.ones(3) / 3, mode="same")

## 1. A step

`c.arms.set()` asks for the new pose at once. The robot never jumps: arm
joints move at most **2 rad/s** (the robot's own limit). Let's see it.

In [ ]:
with r.control() as c:
    c.arms.set(right=home)
    time.sleep(1.0)
    log = JointLog(r)
    t0 = time.time()
    c.arms.set(right=forward)                # the step
    time.sleep(2.0)
    t, q, _ = log.stop(t0)
    c.arms.move_to(right=home, duration=2.0)

v = velocity(t, q[:, PITCH])
print(f"peak speed {np.abs(v).max():.2f} rad/s")

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].step([-0.2, 0, 2], np.degrees([home[0], forward[0], forward[0]]), where="post", label="asked for")
ax[0].plot(t, np.degrees(q[:, PITCH]), label="measured")
ax[0].set_ylabel("shoulder pitch (deg)"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(t, v); ax[1].axhline(-2.0, ls="--", c="gray"); ax[1].text(1.0, -1.9, "limit 2 rad/s")
ax[1].set_ylabel("speed (rad/s)"); ax[1].set_xlabel("time after the step (s)"); ax[1].grid(alpha=.3)
plt.show()

The speed climbs to the limit, holds near it, then drops off sharply as the arm arrives. The acceleration is largest at those two corners, which is hard on the motors and shakes anything the hand is holding.

## 2. Minimum jerk

A smoother way: a path whose speed and acceleration are zero at both ends.
With *s* = time / duration (0 to 1), the minimum-jerk blend is

  x(s) = 10 s³ − 15 s⁴ + 6 s⁵

and the pose is `start + x(s) · (goal − start)`. Here it is in code, checked
against `g1lab.min_jerk`, which `move_to` uses.

In [ ]:
def min_jerk(s):
    s = min(max(s, 0.0), 1.0)
    return 10 * s**3 - 15 * s**4 + 6 * s**5

In [ ]:
s = np.linspace(0, 1, 101)
assert np.allclose([min_jerk(x) for x in s], [g1lab.min_jerk(x) for x in s])
print("matches g1lab.min_jerk")

Now stream it yourself: compute the pose 50 times a second and send it with
`set()`. We keep what we sent, to compare with what the arm did.

In [ ]:
def stream(c, start, goal, duration):
    """Send a minimum-jerk move at RATE per second. Returns (t, sent pitch)."""
    sent = []
    t0 = time.time()
    while True:
        now = time.time() - t0
        pose = start + min_jerk(now / duration) * (goal - start)
        c.arms.set(right=pose)
        sent.append((now, pose[0]))
        if now >= duration:
            break
        time.sleep(1.0 / RATE)
    return np.array(sent)

def run(duration):
    """One move home -> forward over `duration` s; returns sent and measured."""
    with r.control() as c:
        c.arms.move_to(right=home, duration=1.5)
        time.sleep(0.5)
        log = JointLog(r)
        t0 = time.time()
        sent = stream(c, home, forward, duration)
        time.sleep(1.0)                          # let the arm arrive
        t, q, _ = log.stop(t0)
        c.arms.move_to(right=home, duration=1.5)
    return sent, t, q[:, PITCH]

sent, t, pitch = run(2.0)
plt.figure(figsize=(8, 3))
plt.plot(sent[:, 0], np.degrees(sent[:, 1]), label="sent")
plt.plot(t, np.degrees(pitch), label="measured")
plt.xlabel("time (s)"); plt.ylabel("shoulder pitch (deg)"); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 3. Tracking error, and where the rate limit takes over

The **tracking error** is what you sent minus what the arm did, at the same
moment. The arm lags, so the error is largest where the path is fastest. The
minimum-jerk path's top speed is 1.875 · distance / duration, so the next cell
works out the duration at which a 0.8 rad move reaches the 2 rad/s limit. The
cell after it runs shorter and shorter moves to see what happens there.

In [ ]:
# The shortest duration the rate limit lets through untouched.
distance = abs(forward[0] - home[0])
print(f"top speed = 2 rad/s at a duration of {1.875 * distance / 2.0:.2f} s")

In [ ]:
results = []
for duration in (2.0, 1.2, 0.8, 0.5, 0.3):
    sent, t, pitch = run(duration)
    ok = t <= duration + 1.0
    asked = np.interp(t[ok], sent[:, 0], sent[:, 1], left=sent[0, 1], right=sent[-1, 1])
    err = asked - pitch[ok]
    top = np.abs(velocity(t, pitch)).max()
    results.append((duration, np.sqrt(np.mean(err**2)), np.abs(err).max(), top))
    print(f"{duration:4.1f} s: rms error {math.degrees(results[-1][1]):5.2f} deg, "
          f"max {math.degrees(results[-1][2]):5.2f} deg, top speed {top:.2f} rad/s")

d = np.array(results)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(d[:, 0], np.degrees(d[:, 2]), "o-"); ax[0].set_xlabel("duration (s)")
ax[0].set_ylabel("max tracking error (deg)"); ax[0].grid(alpha=.3)
ax[1].plot(d[:, 0], d[:, 3], "o-", label="measured")
ax[1].plot(d[:, 0], 1.875 * distance / d[:, 0], "--", label="planned")
ax[1].axhline(2.0, c="gray", ls=":"); ax[1].set_xlabel("duration (s)")
ax[1].set_ylabel("top speed (rad/s)"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.show()

Below that duration the measured top speed stops following the planned one and stays near 2 rad/s, and the error grows faster, because the robot is slowing your motion for you. To keep that from happening, plan each move to last at least 1.875 · distance / 2 seconds.

**Try this.** Set `forward[0] -= 0.4` in the setup cell, work out the new duration, and run part 3 again to see whether the corner moves where you expect.